In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(DelayedArray)
    library(hooke)
})

In [ ]:
aggregate_gene_score = function(cds, gene_list=c("CCNB1", "CCNB2","CDK1"), name = "gene_list_score"){
    cds_gene_list_norm = cds[rowData(cds)$gene_short_name %in% gene_list,] 
    aggregate_marker_expression = counts(cds_gene_list_norm)
    aggregate_marker_expression = t(t(aggregate_marker_expression) / colData(cds_gene_list_norm)$Size_Factor)
    aggregate_marker_expression = Matrix::colSums(aggregate_marker_expression)
    aggregate_marker_expression = log(aggregate_marker_expression + 1)
    colData(cds)[[name]] = aggregate_marker_expression
    return(cds)
}

normalize <- function(x) {
    return((x- min(x)) /(max(x)-min(x)))
}

detect_genes_fixed = function (cds, min_expr = 0) {
    assertthat::assert_that(methods::is(cds, "cell_data_set"))
    assertthat::assert_that(is.numeric(min_expr))
    mat_bin <- counts(cds) > min_expr
    rowData(cds)$num_cells_expressed <- Matrix::rowSums(mat_bin)
    colData(cds)$num_genes_expressed <- Matrix::colSums(mat_bin)
    cds
}


estimate_cell_cycle = function(cds, g1s_markers, g2m_markers){
    cds_g1s = cds[rowData(cds)$gene_short_name %in% g1s_markers,] 
    aggregate_g1s_expression = counts(cds_g1s)
    aggregate_g1s_expression = t(t(aggregate_g1s_expression) / colData(cds_g1s)$Size_Factor)
    aggregate_g1s_expression = Matrix::colSums(aggregate_g1s_expression)

    cds_g2m = cds[rowData(cds)$gene_short_name %in% g2m_markers,] 
    aggregate_g2m_expression = counts(cds_g2m)
    aggregate_g2m_expression = t(t(aggregate_g2m_expression) / colData(cds_g2m)$Size_Factor)
    aggregate_g2m_expression = Matrix::colSums(aggregate_g2m_expression)
    cds$g1s_score = log(aggregate_g1s_expression+1)
    cds$g2m_score = log(aggregate_g2m_expression+1)
    cds$proliferation_index = log(aggregate_g1s_expression + aggregate_g2m_expression + 1)
    return(cds)
}

In [ ]:
cds_VCS = readRDS("../data/cds_VCS_with_cell_type_labels_E15-15.5_1e-4_subclustered_scores_by_embryo_30_all_genes_umap2_final.RDS")

In [ ]:
summary(cds_VCS$RNA_UMIs)

In [ ]:
cds_VCS$Grouping = paste(cds_VCS$Experiment, cds_VCS$Embryo, sep="-")
cds_VCS$Cluster_group = paste0(cds_VCS$trajectory_refined, "_", cds_VCS$Cluster_sub)

In [ ]:
metadata = as.data.frame(colData(cds_VCS))

# Score rank comparison

In [ ]:
metadata_cluster_collapsed = metadata %>%
    mutate(Phenotypes = paste0(Marker, "_", Phenotype)) %>%
    distinct(trajectory_refined, Cluster_sub, Phenotypes, .keep_all=T) %>%
    tidyr::spread(Phenotypes, avg_score, fill = 0) %>%
    group_by(trajectory_refined, Cluster_sub) %>%
    mutate(Nucleolin_Area = sum(Nucleolin_Area),
           Nucleolin_Ratio = sum(Nucleolin_Ratio),
           SC35_Area = sum(SC35_Area),
           SC35_Ratio = sum(SC35_Ratio),
           Nuclear_Area = sum(Nuclear_Area)) %>%
    as.data.frame() %>%
    distinct(trajectory_refined, Cluster_sub, .keep_all=T)

metadata_cluster_collapsed

In [ ]:
metadata_cluster_collapsed <- metadata_cluster_collapsed %>%
    mutate(sort_tiebreaker = paste0(trajectory_refined, '-', Cluster_sub))

rank_cols <- c(
    "Nuclear_area_rank" = "Nuclear_Area", 
    "Nucleolar_area_rank" = "Nucleolin_Area", 
    "Nucleolar_fraction_rank" = "Nucleolin_Ratio", 
    "SC35_area_rank" = "SC35_Area", 
    "SC35_fraction_rank" = "SC35_Ratio"
)

for(out_col in names(rank_cols)) {
    in_col <- rank_cols[[out_col]]
    metadata_cluster_collapsed <- metadata_cluster_collapsed %>%
        arrange(.data[[in_col]], sort_tiebreaker) %>%
        mutate(!!out_col := row_number())
}

metadata_cluster_collapsed <- metadata_cluster_collapsed %>% select(-sort_tiebreaker)

In [ ]:
plot_and_correlate <- function(df, var_x, var_y, rank_x, rank_y, plot_xlab, plot_ylab) {
    
    clean_df <- df %>% filter(.data[[var_x]] > 0 & .data[[var_y]] > 0)
    
    cor_stats <- clean_df %>%
        summarise(
            spearman_rho = cor(.data[[rank_x]], .data[[rank_y]], method = "spearman", use = "complete.obs"),
            pearson_r    = cor(.data[[rank_x]], .data[[rank_y]], method = "pearson", use = "complete.obs"),
            n_points     = n()
        )
    #print(paste(var_x, var_y))
    print(cor_stats)
    
    
    # 3. Generate Plot
    p <- clean_df %>%
        ggplot(aes(x = .data[[rank_x]], y = .data[[rank_y]])) + 
            geom_point(size = 1) + 
            geom_abline(slope = 1) + 
            labs(x = plot_xlab, y = plot_ylab) + 
            monocle3:::monocle_theme_opts() + 
            theme(
                axis.text.x = element_text(color = "black", size = 15),
                axis.text.y = element_text(color = "black", size = 15),
                axis.title = element_text(size = 15),
                legend.position = "none"
        )
    
    print(p)
}

In [ ]:
comparisons <- list(
  # 1. Nuclear_Area vs Rest
  list("Nuclear_Area", "Nucleolin_Area", "Nuclear_area_rank", "Nucleolar_area_rank", "Nuclear area score rank", "Nucleolar area score rank"),
  list("Nuclear_Area", "Nucleolin_Ratio", "Nuclear_area_rank", "Nucleolar_fraction_rank", "Nuclear area score rank", "Nucleolar fraction score rank"),
  list("Nuclear_Area", "SC35_Area", "Nuclear_area_rank", "SC35_area_rank", "Nuclear area score rank", "Speckle area score rank"),
  list("Nuclear_Area", "SC35_Ratio", "Nuclear_area_rank", "SC35_fraction_rank", "Nuclear area score rank", "Speckle fraction score rank"),
  
  # 2. Nucleolin_Area vs Rest
  list("Nucleolin_Area", "Nucleolin_Ratio", "Nucleolar_area_rank", "Nucleolar_fraction_rank", "Nucleolar area score rank", "Nucleolar fraction score rank"),
  list("Nucleolin_Area", "SC35_Area", "Nucleolar_area_rank", "SC35_area_rank", "Nucleolar area score rank", "Speckle area score rank"),
  list("Nucleolin_Area", "SC35_Ratio", "Nucleolar_area_rank", "SC35_fraction_rank", "Nucleolar area score rank", "Speckle fraction score rank"),
  
  # 3. Nucleolin_Ratio vs Rest
  list("Nucleolin_Ratio", "SC35_Area", "Nucleolar_fraction_rank", "SC35_area_rank", "Nucleolar fraction score rank", "Speckle area score rank"),
  list("Nucleolin_Ratio", "SC35_Ratio", "Nucleolar_fraction_rank", "SC35_fraction_rank", "Nucleolar fraction score rank", "Speckle fraction score rank"),
  
  # 4. SC35_Area vs Rest
  list("SC35_Area", "SC35_Ratio", "SC35_area_rank", "SC35_fraction_rank", "Speckle area score rank", "Speckle fraction score rank")
)

for (comp in comparisons) {
  cat("\n----------------------------------------\n")
  cat(sprintf("%s vs %s\n", comp[[1]], comp[[2]]))
  cat("----------------------------------------\n")

  plot_and_correlate(
    metadata_cluster_collapsed, 
    comp[[1]], comp[[2]], 
    comp[[3]], comp[[4]], 
    comp[[5]], comp[[6]]
  )
}